In [5]:
import os
import json
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, Subset

# -------- Dataset（160次元） --------
class RelativeSpeedDataset160D(Dataset):
    def __init__(self, annot_root, distance_json_path, max_items=None):
        self.items = []
        with open(distance_json_path, encoding='utf-8') as f:
            self.distances = json.load(f)

        for fname in sorted(os.listdir(annot_root)):
            if not fname.endswith(".json"):
                continue
            sid = fname.replace(".json", "")
            if sid not in self.distances:
                continue

            with open(os.path.join(annot_root, fname), encoding='utf-8') as f:
                ann = json.load(f)
            seq = ann['sequence']
            if len(seq) < 20:
                continue

            own = np.array([f['OwnSpeed'] for f in seq], dtype=np.float32)
            tgt = np.array([f['TgtSpeed_ref'] for f in seq], dtype=np.float32)
            keys = sorted(self.distances[sid].keys())
            if len(keys) < 20:
                continue
            dist = np.array([self.distances[sid][k] for k in keys], dtype=np.float32)

            def smooth(x, w):
                return np.convolve(x, np.ones(w)/w, mode='same') if len(x) >= w else np.zeros_like(x)

            for i in range(len(seq) - 19):
                if max_items and len(self.items) >= max_items:
                    return

                d = dist[i:i+20]
                o = own[i:i+20]
                t = tgt[i:i+20]
                if np.any(np.isnan(d)) or np.any(np.isnan(o)) or np.any(np.isnan(t)):
                    continue

                rel_speed = t - o
                acc = np.gradient(o)
                d1 = np.gradient(d)
                d2 = np.gradient(d1)
                f3 = smooth(d, 3)
                f5 = smooth(d, 5)
                f11 = smooth(d, 11)

                try:
                    feat = np.concatenate([
                        d, o, acc, d1, d2,
                        f3[:20], f5[:20], f11[:20]
                    ])
                except:
                    continue

                if feat.shape[0] != 160:
                    continue

                target = np.mean(rel_speed)
                self.items.append((feat.astype(np.float32), target, sid))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        feat, tgt, sid = self.items[idx]
        return torch.tensor(feat), torch.tensor(tgt, dtype=torch.float32), sid

# -------- モデル定義 --------
class AttnLSTMv4Model(nn.Module):
    def __init__(self, input_dim=8, hidden_dim=128, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)

        self.attn_fc = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.Softplus(),
            nn.Linear(128, 64),
            nn.Softplus(),
            nn.Linear(64, 1)
        )

        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.Softplus(),
            nn.Linear(128, 64),
            nn.Softplus(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = x.view(x.size(0), 20, 8)         # (B, 20, 8)
        lstm_out, _ = self.lstm(x)           # (B, 20, hidden)
        attn_weights = torch.softmax(self.attn_fc(lstm_out), dim=1)  # (B, 20, 1)
        context = (lstm_out * attn_weights).sum(dim=1)               # (B, hidden)
        return self.fc(context).squeeze(1)

# -------- 学習ループ --------
def train_attn_lstm_v4(dataset, save_path="model_attn_lstm_v4.pth"):
    scenes = sorted(set([item[-1] for item in dataset.items]))
    train_scenes, val_scenes = train_test_split(scenes, test_size=0.2, random_state=42)
    train_idx = [i for i, item in enumerate(dataset.items) if item[-1] in train_scenes]
    val_idx = [i for i, item in enumerate(dataset.items) if item[-1] in val_scenes]

    train_ds = Subset(dataset, train_idx)
    val_ds = Subset(dataset, val_idx)

    def collate_fn(batch):
        feats, tgts, _ = zip(*batch)
        return torch.stack(feats), torch.tensor(tgts)

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, collate_fn=collate_fn)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = AttnLSTMv4Model().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.SmoothL1Loss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    best_val_loss = float('inf')
    patience = 30
    min_delta = 0.0002
    counter = 0

    for epoch in range(100):
        model.train()
        total_train_loss = 0
        for feats, tgts in tqdm(train_loader, desc=f"[Train {epoch+1}]"):
            feats, tgts = feats.to(device), tgts.to(device)
            optimizer.zero_grad()
            loss = criterion(model(feats), tgts)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item() * feats.size(0)

        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for feats, tgts in val_loader:
                feats, tgts = feats.to(device), tgts.to(device)
                loss = criterion(model(feats), tgts)
                total_val_loss += loss.item() * feats.size(0)

        train_loss = total_train_loss / len(train_ds)
        val_loss = total_val_loss / len(val_ds)
        scheduler.step(val_loss)

        print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        if best_val_loss - val_loss > min_delta:
            best_val_loss = val_loss
            torch.save(model.state_dict(), save_path)
            print(f"✅ Saved model to {save_path} (val_loss={val_loss:.4f})")
            counter = 0
        else:
            counter += 1
            print(f"⏸ No improvement. Patience: {counter}/{patience}")
            if counter >= patience:
                print(f"🛑 Early stopping at epoch {epoch+1}")
                break

    return model

# -------- 実行 --------
if __name__ == "__main__":
    dataset = RelativeSpeedDataset160D(
        annot_root="../train/train_annotations",
        distance_json_path="../distance_ref_data.json",
        max_items=7500
    )

    model = train_attn_lstm_v4(dataset, save_path="model_attn_lstm_v4.pth")
    print("✅ 学習完了: model_attn_lstm_v4.pth に保存しました")


[Train 1]: 100%|██████████| 93/93 [00:01<00:00, 92.32it/s] 


Epoch 1 | Train Loss: 2.8371 | Val Loss: 1.5985
✅ Saved model to model_attn_lstm_v4.pth (val_loss=1.5985)


[Train 2]: 100%|██████████| 93/93 [00:00<00:00, 199.25it/s]


Epoch 2 | Train Loss: 0.3303 | Val Loss: 0.2424
✅ Saved model to model_attn_lstm_v4.pth (val_loss=0.2424)


[Train 3]: 100%|██████████| 93/93 [00:00<00:00, 201.29it/s]


...

Epoch 92 | Train Loss: 0.0076 | Val Loss: 0.0456
⏸ No improvement. Patience: 28/30


[Train 93]: 100%|██████████| 93/93 [00:00<00:00, 200.93it/s]


Epoch 93 | Train Loss: 0.0075 | Val Loss: 0.0460
⏸ No improvement. Patience: 29/30


[Train 94]: 100%|██████████| 93/93 [00:00<00:00, 203.08it/s]


Epoch 94 | Train Loss: 0.0075 | Val Loss: 0.0454
⏸ No improvement. Patience: 30/30
🛑 Early stopping at epoch 94
✅ 学習完了: model_attn_lstm_v4.pth に保存しました


In [6]:
import os
import json
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
from tqdm import tqdm

# -------- モデル定義（学習時と同じ構造） --------
class AttnLSTMv4Model(nn.Module):
    def __init__(self, input_dim=8, hidden_dim=128, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)

        self.attn_fc = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.Softplus(),
            nn.Linear(128, 64),
            nn.Softplus(),
            nn.Linear(64, 1)
        )

        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.Softplus(),
            nn.Linear(128, 64),
            nn.Softplus(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = x.view(x.size(0), 20, 8)
        lstm_out, _ = self.lstm(x)
        attn_weights = torch.softmax(self.attn_fc(lstm_out), dim=1)
        context = (lstm_out * attn_weights).sum(dim=1)
        return self.fc(context).squeeze(1)

# -------- 推論用 Dataset（特徴量160D = 8次元×20フレーム） --------
class InferenceDataset160D(Dataset):
    def __init__(self, annot_root, distance_json_path):
        self.items = []
        self.seq_lens = {}

        with open(distance_json_path, encoding='utf-8') as f:
            self.distances = json.load(f)

        for fname in sorted(os.listdir(annot_root)):
            if not fname.endswith(".json"):
                continue
            sid = fname.replace(".json", "")
            if sid not in self.distances:
                continue

            with open(os.path.join(annot_root, fname), encoding='utf-8') as f:
                ann = json.load(f)
            seq = ann['sequence']
            self.seq_lens[sid] = len(seq)

            if len(seq) < 20:
                continue

            own = np.array([f['OwnSpeed'] for f in seq], dtype=np.float32)
            keys = sorted(self.distances[sid].keys())
            if len(keys) < 20:
                continue
            dist = np.array([self.distances[sid][k] for k in keys], dtype=np.float32)

            def smooth(x, w):
                return np.convolve(x, np.ones(w)/w, mode='same') if len(x) >= w else np.zeros_like(x)

            for i in range(len(seq) - 19):
                d = dist[i:i+20]
                o = own[i:i+20]
                if np.any(np.isnan(d)) or np.any(np.isnan(o)):
                    continue

                acc = np.gradient(o)
                d1 = np.gradient(d)
                d2 = np.gradient(d1)
                f3 = smooth(d, 3)
                f5 = smooth(d, 5)
                f11 = smooth(d, 11)

                feat = np.concatenate([
                    d, o, acc, d1, d2,
                    f3[:20], f5[:20], f11[:20]
                ])

                if feat.shape[0] != 160:
                    continue

                own_avg = np.mean(o)
                self.items.append((feat.astype(np.float32), own_avg, sid, i))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        feat, own_avg, sid, frame_idx = self.items[idx]
        return torch.tensor(feat), own_avg, sid, frame_idx

# -------- 推論 + submission.json 作成 --------
def predict_and_save_submission(
    model_path,
    annot_root,
    distance_json_path,
    save_path="submission.json"
):
    dataset = InferenceDataset160D(annot_root, distance_json_path)
    loader = DataLoader(dataset, batch_size=64, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = AttnLSTMv4Model().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    raw_preds = defaultdict(list)
    with torch.no_grad():
        for feats, own_speeds, sids, frame_idxs in tqdm(loader):
            feats = feats.to(device).view(-1, 20, 8)  # reshape
            preds = model(feats).cpu().numpy()
            own_speeds = own_speeds.numpy()
            abs_speeds = preds + own_speeds  # 相対速度 + 自車速度

            for sid, frame_idx, tgt in zip(sids, frame_idxs, abs_speeds):
                raw_preds[sid].append((frame_idx + 19, float(round(tgt, 3))))  # 20フレーム目の位置に

    submission = {}
    for sid, pairs in raw_preds.items():
        pairs.sort()
        seq_len = dataset.seq_lens.get(sid, max(f for f, _ in pairs) + 1)
        pred_list = [0.0] * seq_len
        for idx, val in pairs:
            if idx < seq_len:
                pred_list[idx] = val
        for i in range(1, seq_len):
            if pred_list[i] == 0.0:
                pred_list[i] = pred_list[i-1]
        submission[sid] = pred_list

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(submission, f, ensure_ascii=False, indent=2)

    print(f"✅ 完成: {save_path} に保存しました（scene数: {len(submission)}）")

# -------- 実行部 --------
if __name__ == "__main__":
    predict_and_save_submission(
        model_path="model_attn_lstm_v4.pth",
        annot_root="../test2/test_annotations",
        distance_json_path="../testdistance/test_spline_smoothed_fixed.json",
        save_path="submission.json"
    )


100%|██████████| 395/395 [00:01<00:00, 303.54it/s]


✅ 完成: submission.json に保存しました（scene数: 239）
